In [202]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [203]:
df = pd.read_csv('data/messy_linear_regression_data.csv')
df.head()


,Age,Experience_Years,Salary,City,Education_Level,Performance_Score,Working_Hours_Per_Week,Random_Noise_Text
0,20.0,21.0,81086.0,Chennai,Bachelors,7.504843403460779,44.0,NaN
1,35.0,30.0,35674.0,Delhi,PhD,7.291322654303698,50.0,rsFUOEbM
2,43.0,15.0,56105.0,Bhubaneswar,PhD,2.715327982680152,69.0,NaN
3,28.0,14.0,111618.0,Mumbai,Bachelors,8.521594893906492,62.0,NaN
4,23.0,27.0,132053.0,Chennai,NaN,unknown,26.0,peVYSkBj


In [204]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13536 entries, 0 to 13535
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Age                     12877 non-null  str    
 1   Experience_Years        12858 non-null  float64
 2   Salary                  12866 non-null  float64
 3   City                    12875 non-null  str    
 4   Education_Level         12886 non-null  str    
 5   Performance_Score       12878 non-null  str    
 6   Working_Hours_Per_Week  12857 non-null  float64
 7   Random_Noise_Text       4112 non-null   str    
dtypes: float64(3), str(5)
memory usage: 846.1 KB


In [205]:
df = df.drop(columns=["Random_Noise_Text"])
df = df.dropna(subset=["Salary"])


In [206]:
df.isnull().sum()

Age                       621
Experience_Years          649
Salary                      0
City                      633
Education_Level           621
Performance_Score         622
Working_Hours_Per_Week    646
dtype: int64

In [207]:
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

In [208]:
df["Performance_Score"] = pd.to_numeric(df["Performance_Score"], errors="coerce")

In [209]:
df["Performance_Score"]=df["Performance_Score"].fillna(df["Performance_Score"].median())
df['Working_Hours_Per_Week']=df['Working_Hours_Per_Week'].fillna(df["Working_Hours_Per_Week"].median())
df["Education_Level"]=df["Education_Level"].fillna(df["Education_Level"].mode()[0])
df['City']=df['City'].fillna(df["City"].mode()[0])
df = df[df["Experience_Years"] <= df["Age"]]

In [210]:
df.head()

,Age,Experience_Years,Salary,City,Education_Level,Performance_Score,Working_Hours_Per_Week
1,35.0,30.0,35674.0,Delhi,PhD,7.291323,50.0
2,43.0,15.0,56105.0,Bhubaneswar,PhD,2.715328,69.0
3,28.0,14.0,111618.0,Mumbai,Bachelors,8.521595,62.0
6,42.0,24.0,76714.0,Bhubaneswar,masters,2.086179,27.0
7,42.0,31.0,144083.0,Mumbai,PhD,4.632194,28.0


In [211]:
df["Education_Level"] = df["Education_Level"].str.strip().str.lower()
df["City"] = df["City"].str.strip().str.lower()
df = pd.get_dummies(df, columns=["City", "Education_Level"],dtype=int, drop_first=True)

In [212]:
df.isnull().sum()
# df.dtypes

Age                        0
Experience_Years           0
Salary                     0
Performance_Score          0
Working_Hours_Per_Week     0
City_chennai               0
City_delhi                 0
City_kolkata               0
City_mumbai                0
Education_Level_diploma    0
Education_Level_masters    0
Education_Level_phd        0
dtype: int64

In [213]:
df.head()

,Age,Experience_Years,Salary,Performance_Score,Working_Hours_Per_Week,City_chennai,City_delhi,City_kolkata,City_mumbai,Education_Level_diploma,Education_Level_masters,Education_Level_phd
1,35.0,30.0,35674.0,7.291323,50.0,0,1,0,0,0,0,1
2,43.0,15.0,56105.0,2.715328,69.0,0,0,0,0,0,0,1
3,28.0,14.0,111618.0,8.521595,62.0,0,0,0,1,0,0,0
6,42.0,24.0,76714.0,2.086179,27.0,0,0,0,0,0,1,0
7,42.0,31.0,144083.0,4.632194,28.0,0,0,0,1,0,0,1


In [227]:
print(df.corr()["Salary"].sort_values(ascending=False))

Salary                     1.000000
City_chennai               0.016975
Education_Level_masters    0.012419
Experience_Years           0.009210
City_kolkata               0.008508
Performance_Score          0.001061
Age                        0.000721
City_mumbai               -0.005267
Working_Hours_Per_Week    -0.007814
City_delhi                -0.008186
Education_Level_phd       -0.009772
Education_Level_diploma   -0.018118
Name: Salary, dtype: float64


In [214]:
x = df.drop(columns=["Salary"])
y = df["Salary"]

In [215]:
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.3, random_state=42)

In [216]:
# from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler()
# xtrain= scaler.fit_transform(xtrain)
# xtest = scaler.transform(xtest)

In [217]:
LinearRegModel = LinearRegression()
LinearRegModel.fit(xtrain, ytrain)


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [218]:
ypred = LinearRegModel.predict(xtest)


In [219]:
print('training score:', LinearRegModel.score(xtrain, ytrain))
print('testing score:', LinearRegModel.score(xtest, ytest))

training score: 0.001066995503225865
testing score: 0.0009521945768687923


### xgboost regressor

In [220]:
from xgboost import XGBRegressor
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1],
}
model=XGBRegressor(objective="reg:squarederror",
    random_state=42)
grid=GridSearchCV(estimator=model,param_grid=param_grid, cv=5)
grid.fit(xtrain,ytrain)


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.01, 0.1], 'max_depth': [3, 5, ...], 'n_estimators': [100, 200]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the 

In [221]:
print('best parameter: ',grid.best_params_)
print('Best score', grid.best_score_)

best parameter:  {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100}
Best score -0.004275554219947297


In [222]:

xtgbrmodel=XGBRegressor(n_estimators=100, learning_rate=0.01, max_depth=3, random_state=42)
xtgbrmodel.fit(xtrain, ytrain)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_met

In [223]:
ypred=xtgbrmodel.predict(xtest)

In [224]:
print('training score:', xtgbrmodel.score(xtrain, ytrain))
print('testing score:', xtgbrmodel.score(xtest, ytest))

training score: 0.028354992723652495
testing score: -0.002852944995098028


# Random forest regressor


In [225]:
from sklearn.ensemble import RandomForestRegressor
rfmodel = RandomForestRegressor(n_estimators=300, random_state=42)
rfmodel.fit(xtrain, ytrain)
ypred_rf = rfmodel.predict(xtest)


In [226]:
print('training score:', rfmodel.score(xtrain, ytrain))
print('testing score:', rfmodel.score(xtest, ytest))

training score: 0.8460574271106054
testing score: -0.04885394139537835


In [228]:
from sklearn.dummy import DummyRegressor

dummy = DummyRegressor(strategy="mean")
dummy.fit(xtrain, ytrain)

print("Dummy Train Score:", dummy.score(xtrain, ytrain))
print("Dummy Test Score:", dummy.score(xtest, ytest))

Dummy Train Score: 0.0
Dummy Test Score: -0.00029156109991501644


In [229]:
dff = pd.read_csv('data/messy_linear_regression_data.csv')
dff.head()

,Age,Experience_Years,Salary,City,Education_Level,Performance_Score,Working_Hours_Per_Week,Random_Noise_Text
0,20.0,21.0,81086.0,Chennai,Bachelors,7.504843403460779,44.0,NaN
1,35.0,30.0,35674.0,Delhi,PhD,7.291322654303698,50.0,rsFUOEbM
2,43.0,15.0,56105.0,Bhubaneswar,PhD,2.715327982680152,69.0,NaN
3,28.0,14.0,111618.0,Mumbai,Bachelors,8.521594893906492,62.0,NaN
4,23.0,27.0,132053.0,Chennai,NaN,unknown,26.0,peVYSkBj


In [230]:
dff.dropna(inplace=True)

In [231]:
dff.info()

<class 'pandas.DataFrame'>
Index: 2924 entries, 1 to 13532
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Age                     2924 non-null   str    
 1   Experience_Years        2924 non-null   float64
 2   Salary                  2924 non-null   float64
 3   City                    2924 non-null   str    
 4   Education_Level         2924 non-null   str    
 5   Performance_Score       2924 non-null   str    
 6   Working_Hours_Per_Week  2924 non-null   float64
 7   Random_Noise_Text       2924 non-null   str    
dtypes: float64(3), str(5)
memory usage: 205.6 KB


In [232]:
dff = dff.drop(columns=["Random_Noise_Text"])
dff["Performance_Score"] = pd.to_numeric(dff["Performance_Score"], errors="coerce")
dff["Age"] = pd.to_numeric(dff["Age"], errors="coerce")
dff = dff.dropna()
dff.isna().sum()

Age                       0
Experience_Years          0
Salary                    0
City                      0
Education_Level           0
Performance_Score         0
Working_Hours_Per_Week    0
dtype: int64

In [233]:
dff.info()


<class 'pandas.DataFrame'>
Index: 2799 entries, 1 to 13532
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Age                     2799 non-null   float64
 1   Experience_Years        2799 non-null   float64
 2   Salary                  2799 non-null   float64
 3   City                    2799 non-null   str    
 4   Education_Level         2799 non-null   str    
 5   Performance_Score       2799 non-null   float64
 6   Working_Hours_Per_Week  2799 non-null   float64
dtypes: float64(5), str(2)
memory usage: 174.9 KB


In [234]:
dff["Education_Level"] = dff["Education_Level"].str.strip().str.lower()
dff["City"] = dff["City"].str.strip().str.lower()
dff = pd.get_dummies(dff, columns=["City", "Education_Level"],dtype=int, drop_first=True)
dff.head()



,Age,Experience_Years,Salary,Performance_Score,Working_Hours_Per_Week,City_chennai,City_delhi,City_kolkata,City_mumbai,Education_Level_diploma,Education_Level_masters,Education_Level_phd
1,35.0,30.0,35674.0,7.291323,50.0,0,1,0,0,0,0,1
6,42.0,24.0,76714.0,2.086179,27.0,0,0,0,0,0,1,0
10,66.0,27.0,83711.0,6.171607,58.0,0,0,0,0,0,1,0
12,35.0,18.0,47916.0,6.956210,28.0,0,1,0,0,0,1,0
16,65.0,15.0,52916.0,5.363320,20.0,1,0,0,0,1,0,0


In [235]:
model = LinearRegression()
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.3, random_state=42)
model.fit(xtrain, ytrain)


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [236]:
print('training score:', model.score(xtrain, ytrain))
print('testing score:', model.score(xtest, ytest))

training score: 0.001066995503225865
testing score: 0.0009521945768687923


In [237]:
from sklearn.ensemble import RandomForestRegressor
rfmodel = RandomForestRegressor(n_estimators=300, random_state=42)
rfmodel.fit(xtrain, ytrain)
ypred_rf = rfmodel.predict(xtest)

In [238]:
print('training score:', rfmodel.score(xtrain, ytrain))
print('testing score:', rfmodel.score(xtest, ytest))

training score: 0.8460574271106054
testing score: -0.04885394139537835
